# MS1 Window-level Contrastive Learning (Minimal Preprocessing Aligned)

This notebook is adapted for the **minimal preprocessing** dataset:

- uses `rt_grid` / `signal_grid`
- keeps `window_size=500`, `stride=250`, `jitter_max=10` in **points**
- uses **unsupervised** file-level positives for SupCon
- uses lighter augmentation for the minimal-input setting
- adds stronger reproducibility controls (seeded RNG in dataset / augment / slicer)


In [ ]:
# --- Imports + core classes (minimal-preprocessing aligned) ---
import os, re
from pathlib import Path
import math
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint

# -------------------------
# Global config
# -------------------------


SEED = 0
# PROJECT_ROOT = Path("../..").resolve()
# META_PATH=PROJECT_ROOT / "data" / "processed" / "metadata_with_frog.csv"
# NPZ_DIR=PROJECT_ROOT / "data" / "processed" /"mzML_npz_45"

def seed_everything_all(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    L.seed_everything(seed, workers=True)

seed_everything_all(SEED)

meta = pd.read_csv(META_PATH, dtype=str)
uhm2trt = dict(zip(meta["UHM_sample"], meta["treatment"]))

def normalize_uhm_from_dataset_filename(fn: str) -> str:
    s = os.path.basename(str(fn))
    s = re.sub(r"\.(npz|mzML)$", "", s)
    s = re.sub(r"_MS1$", "", s)
    return s

def file_name_to_treatment(file_name: str) -> str:
    uhm = normalize_uhm_from_dataset_filename(file_name)
    trt = uhm2trt.get(uhm, None)
    if trt is None:
        raise KeyError(f"UHM '{uhm}' not found in metadata. file_name={file_name}")
    return trt


class ChromAugment:
    """
    Conservative LC-MS augmentation for the minimal-preprocessing data.
    Signal only; RT is kept unchanged.

    Notes
    -----
    - Uses its own RNG for reproducibility.
    - p95 is used as a robust intensity reference.
    """
    def __init__(
        self,
        amp_scale_range=(0.9, 1.1),
        baseline_offset_frac=0.0,
        noise_frac=0.005,
        clip_min=0.0,
        seed: int = 0,
    ):
        self.amp_scale_range = tuple(amp_scale_range)
        self.baseline_offset_frac = float(baseline_offset_frac)
        self.noise_frac = float(noise_frac)
        self.clip_min = clip_min
        self.rng = np.random.default_rng(seed)

    def __call__(self, chromatogram: dict) -> dict:
        rt = chromatogram["rt"]
        sig = chromatogram["signal"]

        if hasattr(rt, "cpu"):
            rt = rt.cpu().numpy()
        if hasattr(sig, "cpu"):
            sig = sig.cpu().numpy()

        rt = np.asarray(rt, dtype=np.float32)
        sig = np.asarray(sig, dtype=np.float32)

        p95 = float(np.percentile(np.abs(sig), 95)) + 1e-12

        a = float(self.rng.uniform(*self.amp_scale_range))
        sig2 = sig * a

        if self.baseline_offset_frac > 0:
            b = float(self.rng.uniform(-self.baseline_offset_frac, self.baseline_offset_frac)) * p95
            sig2 = sig2 + b

        if self.noise_frac > 0:
            noise_std = self.noise_frac * p95
            sig2 = sig2 + self.rng.normal(0.0, noise_std, size=sig2.shape).astype(np.float32)

        if self.clip_min is not None:
            sig2 = np.maximum(sig2, self.clip_min).astype(np.float32)

        out = dict(chromatogram)
        out["rt"] = rt.astype(np.float32)
        out["signal"] = sig2.astype(np.float32)
        return out


class WindowSlicer:
    """
    Window slicing in POINTS (not seconds).

    Current convention:
    - window_size=500 points
    - stride=250 points
    - jitter_max=10 points

    Since the minimal dataset is already on a shared RT grid (~0.2 sec / point),
    these point-based settings are now stable across files.
    """
    def __init__(self, window_size=500, stride=250, jitter_max=0, seed: int = 0):
        self.window_size = int(window_size)
        self.stride = int(stride)
        self.jitter_max = int(jitter_max)
        self.rng = np.random.default_rng(seed)

    def __call__(self, chromatogram):
        rt = chromatogram["rt"]
        signal = chromatogram["signal"]
        chrom_id = chromatogram.get("chrom_id", None)
        chrom_name = chromatogram.get("chrom_name", None)

        if chrom_id is None:
            raise KeyError("chromatogram must contain 'chrom_id'")

        if hasattr(rt, "cpu"):
            rt = rt.cpu().numpy()
        if hasattr(signal, "cpu"):
            signal = signal.cpu().numpy()

        rt = np.asarray(rt)
        signal = np.asarray(signal)

        Lsig = len(rt)
        windows = []
        start = 0

        j = int(self.rng.integers(-self.jitter_max, self.jitter_max + 1)) if self.jitter_max > 0 else 0

        while start + self.window_size <= Lsig:
            s = start + j
            s = max(0, min(s, Lsig - self.window_size))
            e = s + self.window_size

            windows.append({
                "rt": rt[s:e].astype(np.float32),
                "signal": signal[s:e].astype(np.float32),
                "start": int(s),
                "chrom_id": int(chrom_id),
                "chrom_name": chrom_name,
                "jitter": int(j),
            })
            start += self.stride

        return windows


class MassSpecWindowDataset(torch.utils.data.Dataset):
    """
    Window-level dataset.
    Each MS1 file:
      -> two independent file-level augmentations
      -> slice into windows AFTER augmentation
    """
    def __init__(
        self,
        npz_files,
        window_slicer,
        augment: ChromAugment | None = None,
        n_windows_target: int = 12,
        seed: int = 0,
    ):
        self.npz_files = [Path(p) for p in npz_files]
        self.slicer = window_slicer
        self.augment = augment
        self.n_windows_target = n_windows_target
        self.rng = np.random.default_rng(seed)
        self.file_id_map = {p.name: i for i, p in enumerate(self.npz_files)}

    def __len__(self):
        return len(self.npz_files)

    def _load_chrom(self, npz_path: Path) -> dict:
        d = np.load(npz_path)
        file_id = self.file_id_map[npz_path.name]
        return {
            "rt": d["rt_grid"].astype(np.float32),
            "signal": d["signal_grid"].astype(np.float32),
            "chrom_id": int(file_id),
            "chrom_name": npz_path.name,
        }

    def _sample_windows(self, windows: list[dict]) -> list[dict]:
        n = len(windows)
        k = self.n_windows_target
        if k is None:
            return windows
        if n == 0:
            return []

        if n >= k:
            idx = self.rng.choice(n, size=k, replace=False)
        else:
            idx = self.rng.choice(n, size=k, replace=True)

        return [windows[i] for i in idx]

    def __getitem__(self, idx):
        npz_path = self.npz_files[idx]
        chrom = self._load_chrom(npz_path)

        chromA = self.augment(chrom) if self.augment is not None else chrom
        chromB = self.augment(chrom) if self.augment is not None else chrom

        assert isinstance(chromA, dict) and isinstance(chromB, dict), (type(chromA), type(chromB))

        winsA = self._sample_windows(self.slicer(chromA))
        winsB = self._sample_windows(self.slicer(chromB))

        def pack(wins):
            X = []
            for w in wins:
                sig = w["signal"].astype(np.float32)
                rt = w["rt"].astype(np.float32)
                X.append(np.stack([sig, rt], axis=1))
            return np.stack(X, axis=0)  # (N,L,2)

        XA = pack(winsA)
        XB = pack(winsB)
        chrom_id = chrom["chrom_id"]
        chrom_name = chrom["chrom_name"]
        return {
            "x": np.stack([XA, XB], axis=0),  # (2,N,L,2)
            "file_id": int(chrom_id),
            "file_name": chrom_name,
            "chrom_id": int(chrom_id),
            "chrom_name": chrom_name,
        }


class MassSpecWindowContrastEncoder(nn.Module):
    """
    Window-level encoder:
    Patch(Conv1d) + Transformer + CLS + RT sinusoidal positional encoding.
    Input:  signal(B,L), rt(B,L)
    Output: embedding(B,embed_dim)
    """
    def __init__(self, patch_size=50, stride=50, embed_dim=64, num_heads=4):
        super().__init__()
        self.patch_size = patch_size
        self.stride = stride
        self.embed_dim = embed_dim

        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=stride,
            padding=0,
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.positional_encoding = SinusoidalPositionalEncoding(embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=4 * embed_dim,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

    def forward(self, signal, rt):
        B = signal.size(0)
    
        if not torch.isfinite(signal).all():
            raise RuntimeError("signal contains NaN/Inf before encoder")
        if not torch.isfinite(rt).all():
            raise RuntimeError("rt contains NaN/Inf before encoder")
    
        # light numeric stabilization: keep shape, only shrink magnitude
        signal = signal / 1e6
    
        x = self.conv(signal.unsqueeze(1))   # (B, embed_dim, n_patches)
        if not torch.isfinite(x).all():
            raise RuntimeError("conv output contains NaN/Inf")
    
        x = x.permute(0, 2, 1)               # (B, n_patches, embed_dim)
    
        rt_patch = rt[:, self.patch_size - 1::self.stride]
        assert x.size(1) == rt_patch.size(1), (
            f"Patch/RT mismatch: conv patches={x.size(1)} vs rt patches={rt_patch.size(1)}. "
            f"patch_size={self.patch_size}, stride={self.stride}, input_len={rt.size(1)}"
        )
    
        rt_patch = rt_patch / 1800.0
        pe = self.positional_encoding(rt_patch)
        if not torch.isfinite(pe).all():
            raise RuntimeError("positional encoding contains NaN/Inf")
    
        x = x + pe
        if not torch.isfinite(x).all():
            raise RuntimeError("x + positional encoding contains NaN/Inf")
    
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
    
        x = self.transformer(x)
        if not torch.isfinite(x).all():
            raise RuntimeError("transformer output contains NaN/Inf")
    
        h = x[:, 0, :]
        #h = x[:, 1:, :].mean(dim=1)
        h = F.layer_norm(h, h.shape[-1:])
        if not torch.isfinite(h).all():
            raise RuntimeError("encoder output h contains NaN/Inf")
    
        return h


class SupConLoss_mean(nn.Module):
    """Stable SupCon (mean over positives) using logsumexp."""
    def __init__(self, temperature=0.5, eps: float = 1e-8):
        super().__init__()
        self.temperature = temperature
        self.eps = float(eps)

    def forward(self, features, labels):
        device = features.device
        N = features.size(0)

        z = F.normalize(features, dim=1, eps=self.eps)
        labels = labels.view(-1, 1).to(device)

        pos_mask = torch.eq(labels, labels.T)
        self_mask = torch.eye(N, device=device, dtype=torch.bool)
        pos_mask = pos_mask & (~self_mask)

        logits = (z @ z.T) / self.temperature

        # 数值稳定：每行减去最大值
        logits_max, _ = logits.max(dim=1, keepdim=True)
        logits = logits - logits_max.detach()

        # 去掉 self
        logits = logits.masked_fill(self_mask, float("-inf"))

        log_denom = torch.logsumexp(logits, dim=1, keepdim=True)
        log_prob = logits - log_denom

        # 只在 positive 位置取值，别让无关位置的 nan/inf 混进来
        pos_count = pos_mask.sum(dim=1).float()
        valid = pos_count > 0

        if not valid.any():
            return torch.zeros([], device=device, dtype=features.dtype, requires_grad=True)

        log_prob_pos = log_prob.masked_fill(~pos_mask, 0.0)
        mean_log_prob_pos = log_prob_pos.sum(dim=1) / (pos_count + self.eps)

        loss = -mean_log_prob_pos[valid].mean()

        if not torch.isfinite(loss):
            raise RuntimeError("SupCon loss became NaN/Inf")

        return loss





class ProjectionHead(nn.Module):
    def __init__(self, in_dim=64, hidden_dim=128, out_dim=64,use_bn=False):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),   # 👈 替换 BN
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

        

class WindowEncoderWithHead(nn.Module):
    """encoder -> h ; projection head -> z"""
    def __init__(self, encoder, proj_hidden=128, proj_out=64, use_bn=False):
        super().__init__()
        self.encoder = encoder

        in_dim = getattr(encoder, "embed_dim", None)
        if in_dim is None:
            raise ValueError("encoder must have attribute `embed_dim`")

        self.proj = ProjectionHead(
            in_dim=in_dim,
            hidden_dim=proj_hidden,
            out_dim=proj_out,
            use_bn=use_bn,
        )

    def forward(self, signal, rt):
        h = self.encoder(signal, rt)
        if not torch.isfinite(h).all():
            raise RuntimeError("h contains NaN/Inf in WindowEncoderWithHead")
    
        z = self.proj(h)
        if not torch.isfinite(z).all():
            raise RuntimeError("z contains NaN/Inf in WindowEncoderWithHead")
    
        return h, z


class MassSpecWindowContrast(L.LightningModule):
    """Window-level contrastive learning."""
    def __init__(
        self,
        encoder: nn.Module,
        temperature: float = 0.5,
        lr: float = 1e-3,
        weight_decay: float = 1e-4,
        log_every_n_steps: int = 10,
    ):
        super().__init__()
        self.encoder = encoder
        self.temperature = temperature
        self.lr = lr
        self.weight_decay = weight_decay
        self.log_every_n_steps = log_every_n_steps
        self.enc_with_head = WindowEncoderWithHead(self.encoder, proj_hidden=128, proj_out=64, use_bn=False)
        self.supcon_loss = SupConLoss_mean(temperature=temperature)
        self.save_hyperparameters(ignore=["encoder"])

    def training_step(self, batch, batch_idx):
        signal = batch["signal"].to(self.device)
        rt = batch["rt"].to(self.device)
        y = batch["y_file"].to(self.device)
    
        h, z = self.enc_with_head(signal, rt)
        loss = self.supcon_loss(z, y)
        self.log("train/loss", loss, prog_bar=True)
    
        with torch.no_grad():
            feat = F.normalize(z, dim=1)
            N = feat.size(0)
    
            std_mean = feat.std(dim=0).mean()
            self.log("debug/std_mean", std_mean, prog_bar=True)
    
            m = min(512, N)
            idx = torch.randperm(N, device=feat.device)[:m]
            feat_sub = feat[idx]
            lbl = y[idx]
    
            sim = feat_sub @ feat_sub.T
            mask_offdiag = ~torch.eye(m, dtype=torch.bool, device=feat.device)
            sim_off = sim[mask_offdiag]
    
            if sim_off.numel() > 0:
                self.log("debug/sim_off_mean", sim_off.mean(), prog_bar=True)
                self.log("debug/sim_off_max", sim_off.max(), prog_bar=True)
    
            sim_for_nn = sim.masked_fill(~mask_offdiag, -1e9)
            nn_idx = sim_for_nn.argmax(dim=1)
            nn_acc = (lbl[nn_idx] == lbl).float().mean()
            self.log("train/nn_top1", nn_acc, prog_bar=True)
    
        with torch.no_grad():
            feat_h = F.normalize(h, dim=1)
            std_h = feat_h.std(dim=0).mean()
            self.log("debug/h_std_mean", std_h)
    
            feat_h_sub = feat_h[idx]
            sim_h = feat_h_sub @ feat_h_sub.T
            mask_h_offdiag = ~torch.eye(m, dtype=torch.bool, device=feat_h.device)
            sim_h_off = sim_h[mask_h_offdiag]
    
            if sim_h_off.numel() > 0:
                self.log("debug/h_sim_off_mean", sim_h_off.mean())
    
        return loss
    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.register_buffer("div_term", div_term)

    def forward(self, rt):
        rt = rt.unsqueeze(-1)  # (B,seq,1)
        pe = torch.zeros(rt.size(0), rt.size(1), self.d_model, device=rt.device)
        pe[:, :, 0::2] = torch.sin(rt * self.div_term)
        pe[:, :, 1::2] = torch.cos(rt * self.div_term)
        return pe


def collate_window_level_with_name(batch):
    """
    Used for:
    - training (SupCon): y_file
    - evaluation (PCA / LDA): file_name
    """
    sigs, rts, y_files, names = [], [], [], []

    for b in batch:
        x = b["x"]   # (2,N,L,2)
        V, N, Lsig, C = x.shape

        signal = x[..., 0].reshape(V * N, Lsig)
        rt = x[..., 1].reshape(V * N, Lsig)

        sigs.append(torch.from_numpy(signal))
        rts.append(torch.from_numpy(rt))

        fid = b.get("file_id", b.get("chrom_id"))
        if fid is not None:
            y = np.full((V * N,), int(fid), dtype=np.int64)
            y_files.append(torch.from_numpy(y))

        name = b.get("file_name", b.get("chrom_name"))
        names.extend([name] * (V * N))

    out = {
        "signal": torch.cat(sigs, dim=0).float(),
        "rt": torch.cat(rts, dim=0).float(),
        "file_name": names,
    }

    if len(y_files) > 0:
        out["y_file"] = torch.cat(y_files, dim=0).long()

    return out


# =========================
# Train (Phase-1)
# Keep the current point-based slicing convention:

# window_size = 500 points
# stride = 250 points
# jitter_max = 10 points
# This notebook uses the minimal preprocessing dataset: data/processed/mzML_npz_45
# =========================



# =========================
# 0) imports
# =========================
from pathlib import Path
import json
import itertools
import pandas as pd
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader


# =========================
# 1) single run
# =========================
def run_phase1_train(
    seed,
    exp_name,
    npz_dir,
    log_root,
    window_size=500,
    stride=250,
    jitter_max=10,
    amp_scale_range=(0.9, 1.1),
    baseline_offset_frac=0.0,
    noise_frac=0.005,
    n_windows_target=32,
    batch_size=32,
    embed_dim=64,
    patch_size=50,
    encoder_stride=50,
    num_heads=4,
    temperature=1.0,
    lr=1e-4,
    max_epochs=300,
    every_n_train_steps=50,
    gradient_clip_val=1.0,
    devices=1,
):
    seed_everything_all(seed)

    npz_dir = Path(npz_dir)
    log_root = Path(log_root)
    log_root.mkdir(parents=True, exist_ok=True)

    npz_files = sorted(npz_dir.glob("*.npz"))
    print("=" * 80)
    print("Experiment:", exp_name)
    print("Seed:", seed)
    print("Total npz:", len(npz_files))
    print("NPZ_DIR:", npz_dir)

    slicer = WindowSlicer(
        window_size=window_size,
        stride=stride,
        jitter_max=jitter_max,
        seed=seed,
    )

    augment = ChromAugment(
        amp_scale_range=amp_scale_range,
        baseline_offset_frac=baseline_offset_frac,
        noise_frac=noise_frac,
        clip_min=0.0,
        seed=seed,
    )

    dataset = MassSpecWindowDataset(
        npz_files=npz_files,
        window_slicer=slicer,
        augment=augment,
        n_windows_target=n_windows_target,
        seed=seed,
    )

    train_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_window_level_with_name,
        drop_last=False,
    )

    model = MassSpecWindowContrast(
        encoder=MassSpecWindowContrastEncoder(
            embed_dim=embed_dim,
            patch_size=patch_size,
            stride=encoder_stride,
            num_heads=num_heads,
        ),
        temperature=temperature,
        lr=lr,
    )

    logger = CSVLogger(save_dir=str(log_root), name=exp_name)

    checkpoint_cb = ModelCheckpoint(
        dirpath=None,
        save_top_k=-1,
        every_n_train_steps=every_n_train_steps,
        filename="step{step}-loss{train/loss:.3f}",
        auto_insert_metric_name=False,
    )

    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=devices,
        gradient_clip_val=gradient_clip_val,
        logger=logger,
        callbacks=[checkpoint_cb],
        log_every_n_steps=1,
    )

    cfg = {
        "seed": seed,
        "exp_name": exp_name,
        "npz_dir": str(npz_dir),
        "log_root": str(log_root),
        "window_size": window_size,
        "stride": stride,
        "jitter_max": jitter_max,
        "amp_scale_range": amp_scale_range,
        "baseline_offset_frac": baseline_offset_frac,
        "noise_frac": noise_frac,
        "n_windows_target": n_windows_target,
        "batch_size": batch_size,
        "embed_dim": embed_dim,
        "patch_size": patch_size,
        "encoder_stride": encoder_stride,
        "num_heads": num_heads,
        "temperature": temperature,
        "lr": lr,
        "max_epochs": max_epochs,
        "every_n_train_steps": every_n_train_steps,
        "gradient_clip_val": gradient_clip_val,
        "devices": devices,
    }

    trainer.fit(model, train_loader)

    ckpt_dir = Path(checkpoint_cb.dirpath)
    print("Saved checkpoints in:", ckpt_dir)

    cfg_path = ckpt_dir.parent / "run_config.json"
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    out = {
        "exp_name": exp_name,
        "seed": seed,
        "ckpt_dir": str(ckpt_dir),
        "config_path": str(cfg_path),
    }
    out.update(cfg)
    return out


# =========================
# 2) same config, multiple seeds
# =========================
def run_phase1_multi_seed(
    exp_base_name,
    seeds,
    npz_dir,
    log_root,
    **train_kwargs,
):
    rows = []

    for seed in seeds:
        exp_name = f"{exp_base_name}_seed{seed}"
        out = run_phase1_train(
            seed=seed,
            exp_name=exp_name,
            npz_dir=npz_dir,
            log_root=log_root,
            **train_kwargs,
        )
        rows.append(out)

    return pd.DataFrame(rows)


# =========================
# 3) grid helper
# =========================
def expand_param_grid(param_grid: dict):
    keys = list(param_grid.keys())
    values = [param_grid[k] if isinstance(param_grid[k], (list, tuple)) else [param_grid[k]] for k in keys]
    rows = []
    for combo in itertools.product(*values):
        rows.append(dict(zip(keys, combo)))
    return rows


def make_config_name(base_name: str, cfg: dict):
    parts = [base_name]
    for k, v in cfg.items():
        if isinstance(v, tuple):
            v = "-".join(map(str, v))
        s = str(v).replace(" ", "").replace("/", "_")
        parts.append(f"{k}-{s}")
    return "__".join(parts)


def run_phase1_grid(
    exp_base_name,
    seeds,
    npz_dir,
    log_root,
    param_grid,
    common_kwargs=None,
):
    common_kwargs = {} if common_kwargs is None else dict(common_kwargs)
    grid = expand_param_grid(param_grid)

    all_rows = []
    print(f"Total grid configs: {len(grid)}")

    for i, cfg in enumerate(grid, start=1):
        cfg_name = make_config_name(exp_base_name, cfg)
        print("\n" + "#" * 100)
        print(f"[GRID {i}/{len(grid)}] {cfg_name}")
        print(json.dumps(cfg, indent=2))

        df_runs = run_phase1_multi_seed(
            exp_base_name=cfg_name,
            seeds=seeds,
            npz_dir=npz_dir,
            log_root=log_root,
            **common_kwargs,
            **cfg,
        )
        df_runs["config"] = cfg_name
        df_runs["grid_params"] = [json.dumps(cfg, sort_keys=True)] * len(df_runs)
        all_rows.append(df_runs)

    if len(all_rows) == 0:
        return pd.DataFrame()

    return pd.concat(all_rows, ignore_index=True)


## Train (Phase-1)

Keep the current point-based slicing convention:

- `window_size = 500` points
- `stride = 250` points
- `jitter_max = 10` points

This notebook uses the minimal preprocessing dataset:
`data/processed/mzML_npz_45`


In [ ]:
# ============================================================
# STRICT NESTED LOGO THRESHOLD TUNING
# based on fixed best checkpoint per seed
# ============================================================

import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
from torch.utils.data import DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
)

# ============================================================
# 0) USER CONFIG
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)




#LOG_ROOT=PROJECT_ROOT /"results"/"runs"/"ACC_threshhold"/"Task1710"/"logs"/"logs_1710_19_1"


# ---- paths ----
PROJECT_ROOT = Path("../..").resolve()
META_PATH=PROJECT_ROOT / "data" / "processed" / "metadata_with_frog.csv"
NPZ_DIR=PROJECT_ROOT / "data" / "processed" /"mzML_npz_45"

BEST_CKPT_DIR = PROJECT_ROOT /"results"/"final"/"Task1717"/"1717_BestCKPT"

OUT_DIR = PROJECT_ROOT /"results"/"runs"/"ACC_threshhold"/"Task1717"/"logs"/"task1717_logs22"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- task ----
POS_CLASS = "STP1717.1"
NEG_CLASS = "control"
# 若要跑另一个任务，可改成：
# POS_CLASS = "STP1710.7"
# NEG_CLASS = "control"

# ---- eval / encoder ----
WINDOW_SIZE = 500
STRIDE = 250

EMBED_DIM = 64
PATCH_SIZE = 50
ENCODER_STRIDE = 50
NUM_HEADS = 4

# ---- classifier ----
LR_C = 0.01
LR_MAX_ITER = 5000
FROG_AGG = "mean"                   # mean / median
THRESHOLD_TUNE_METRIC = "balanced_acc"   # "balanced_acc" or "acc"

# ---- run control ----
SEEDS = list(range(20))
RUN_ONE_SEED = None        # 先单独试一个 seed；全部跑时设为 None
SAVE_EMBED_CACHE = False
EMBED_CACHE_DIR = OUT_DIR / "embed_cache"
EMBED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

VERBOSE_OUTER = True
VERBOSE_INNER = False


# ============================================================
# 1) HELPER: parse / metadata
# ============================================================

def parse_step_from_ckpt_name(s):
    m = re.search(r"step(\d+)-loss", str(s))
    return int(m.group(1)) if m else None

def strip_prefix_any(sd, prefixes):
    for p in prefixes:
        if any(k.startswith(p) for k in sd.keys()):
            out = {k[len(p):]: v for k, v in sd.items() if k.startswith(p)}
            return out, p
    return None, None

def frog_from_uhm_sample(s: str) -> str:
    return str(s).split("-")[0]

def frog_from_npz_name(fn: str) -> str:
    return str(fn).split("-")[0]

def sample_id_from_npz_name(npz_name: str) -> str:
    base = Path(npz_name).name
    m = re.search(r"-(\d+)", base)
    return m.group(1) if m else base

def sample_id_from_meta_filename(meta_filename: str) -> str:
    base = str(meta_filename)
    m = re.search(r"WF22_(\d+)", base)
    return m.group(1) if m else base

def build_npz_to_label_maps(npz_files, meta_csv):
    df = pd.read_csv(meta_csv)
    assert "UHM_sample" in df.columns
    assert "treatment" in df.columns
    assert "filename" in df.columns

    meta_by_id = {}
    for _, row in df.iterrows():
        sid = sample_id_from_meta_filename(row["filename"])
        meta_by_id[str(sid)] = row

    y_str = {}
    g_frog = {}
    uhm_sample = {}
    missing = []

    for p in npz_files:
        sid = sample_id_from_npz_name(p.name)
        if str(sid) not in meta_by_id:
            missing.append(p.name)
            continue

        r = meta_by_id[str(sid)]
        uhm = str(r["UHM_sample"])
        trt = str(r["treatment"])
        frog = frog_from_uhm_sample(uhm)

        y_str[p.name] = trt
        g_frog[p.name] = frog
        uhm_sample[p.name] = uhm

    if len(missing) > 0:
        print(f"[WARN] {len(missing)} npz files not matched. Example:", missing[:5])

    return y_str, g_frog, uhm_sample


# ============================================================
# 2) BUILD EVAL LOADER
# ============================================================

def build_eval_loader(npz_dir, seed=0, window_size=500, stride=250):
    out_dir = Path(npz_dir)
    npz_files = sorted(out_dir.glob("*.npz"))
    print("Total npz:", len(npz_files))

    slicer = WindowSlicer(window_size=window_size, stride=stride, jitter_max=0)

    eval_ds = MassSpecWindowDataset(
        npz_files=npz_files,
        window_slicer=slicer,
        augment=None,
        n_windows_target=None,
        seed=seed,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=256,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_window_level_with_name,
        drop_last=False,
    )
    return npz_files, eval_loader


# ============================================================
# 3) LOAD ENCODER FROM CKPT
# ============================================================

def load_encoder_from_ckpt(
    ckpt_path: str,
    device: str,
    embed_dim=64,
    patch_size=50,
    encoder_stride=50,
    num_heads=4,
):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("state_dict", ckpt)

    load_sd, used_prefix = strip_prefix_any(
        state,
        prefixes=["enc_with_head.encoder.", "encoder.", "model.encoder."]
    )
    if load_sd is None:
        raise KeyError(f"Cannot find encoder prefix. Example keys: {list(state.keys())[:5]}")

    # 注意：这里用你 notebook 里的 encoder 定义
    enc = MassSpecWindowContrastEncoder(
        embed_dim=embed_dim,
        patch_size=patch_size,
        stride=encoder_stride,
        num_heads=num_heads,
    ).to(device)

    missing, unexpected = enc.load_state_dict(load_sd, strict=False)
    enc.eval()

    print("loaded:", ckpt_path)
    print("used_prefix:", used_prefix, "| missing:", len(missing), "| unexpected:", len(unexpected))
    return enc


# ============================================================
# 4) EXTRACT FILE EMBEDDING
# ============================================================

@torch.no_grad()
def extract_E_file_from_encoder(encoder, loader, device, y_file_str_map):
    """
    将 window embedding 平均到 file embedding
    """
    Z_sum = defaultdict(lambda: None)
    Z_cnt = defaultdict(int)
    y_str = {}

    encoder.eval()
    encoder.to(device)

    for batch in loader:
        signal = batch["signal"].to(device)
        rt     = batch["rt"].to(device)
        names  = list(batch["file_name"])

        z = encoder(signal, rt)
        z = z.detach().cpu().numpy().astype(np.float64)

        for zi, fn in zip(z, names):
            fn = str(fn)
            if fn not in y_file_str_map:
                continue
            if Z_sum[fn] is None:
                Z_sum[fn] = zi.copy()
            else:
                Z_sum[fn] += zi
            Z_cnt[fn] += 1
            y_str[fn] = y_file_str_map[fn]

    file_names = sorted(Z_sum.keys())
    E_file = np.stack([Z_sum[n] / max(1, Z_cnt[n]) for n in file_names], axis=0)
    y_file_str = np.array([y_str[n] for n in file_names], dtype=object)
    g_file = np.array([frog_from_npz_name(n) for n in file_names], dtype=str)

    return E_file, y_file_str, g_file, file_names


# ============================================================
# 5) TASK FILTER
# ============================================================

def filter_task_files(E_file, y_file_str, g_file, file_names, pos_class, neg_class="control"):
    keep = np.isin(y_file_str, [pos_class, neg_class])

    E2 = E_file[keep]
    y2_str = y_file_str[keep]
    g2 = g_file[keep]
    fn2 = np.array(file_names, dtype=object)[keep]

    if len(E2) == 0:
        raise ValueError(f"No task files for {pos_class} vs {neg_class}")

    y2 = (y2_str == pos_class).astype(int)

    return E2, y2, y2_str, g2, fn2


# ============================================================
# 6) AGGREGATION / THRESHOLD HELPERS
# ============================================================

def aggregate_by_group(values, groups, mode="mean"):
    buckets = defaultdict(list)
    for v, g in zip(values, groups):
        buckets[g].append(float(v))

    group_ids = list(buckets.keys())

    if mode == "mean":
        agg = np.array([np.mean(buckets[g]) for g in group_ids], dtype=float)
    elif mode == "median":
        agg = np.array([np.median(buckets[g]) for g in group_ids], dtype=float)
    else:
        raise ValueError(mode)

    return agg, np.array(group_ids, dtype=object)

def aggregate_frog_labels(y_file, groups_frog):
    y_mean, frogs = aggregate_by_group(y_file, groups_frog, mode="mean")
    y_frog = (y_mean >= 0.5).astype(int)
    return y_frog, frogs

# def candidate_thresholds_from_probs(probs):
#     probs = np.asarray(probs, dtype=float)
#     uniq = np.unique(np.sort(probs))
#     if len(uniq) == 1:
#         return np.array([uniq[0]], dtype=float)

#     mids = (uniq[:-1] + uniq[1:]) / 2.0
#     eps = 1e-12
#     thresholds = np.concatenate([
#         [uniq[0] - eps],
#         mids,
#         [uniq[-1] + eps],
#     ])
#     return thresholds

def candidate_thresholds_from_probs(probs):
    """
    Generate a dense, uniform grid of candidate thresholds.
    Note: we intentionally do NOT depend on probs to improve stability
    under small-sample LOFO evaluation.
    """
    return np.linspace(0.05, 0.95, 181)

def choose_best_threshold(y_true, probs, metric="balanced_acc"):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs).astype(float)

    thrs = candidate_thresholds_from_probs(probs)

    best_thr = None
    best_score = -np.inf

    for thr in thrs:
        pred = (probs >= thr).astype(int)

        if metric == "balanced_acc":
            score = balanced_accuracy_score(y_true, pred)
        elif metric == "acc":
            score = accuracy_score(y_true, pred)
        else:
            raise ValueError(metric)

        # tie-break: 更接近0.5优先
        if (
            score > best_score
            or (
                np.isclose(score, best_score)
                and best_thr is not None
                and abs(thr - 0.5) < abs(best_thr - 0.5)
            )
        ):
            best_score = score
            best_thr = float(thr)

    return best_thr, float(best_score)


# ============================================================
# 7) STRICT NESTED LOGO FOR ONE TASK
# ============================================================

def run_strict_nested_logo_threshold_tuning(
    E,
    y,
    groups_frog,
    file_names=None,
    C=10.0,
    frog_agg="mean",
    threshold_metric="balanced_acc",
    max_iter=5000,
    verbose_outer=True,
    verbose_inner=False,
):
    """
    严格 nested LOGO:
      outer: hold out 1 frog as test
      inner: within outer-train, LOGO to generate inner OOF frog probs
      threshold: chosen on inner OOF frog probs
      final: refit on full outer-train, predict outer-test once

    输入:
      E            : file embedding, shape (n_files, d)
      y            : file label 0/1
      groups_frog  : frog id for each file
      file_names   : optional
    """
    E = np.asarray(E)
    y = np.asarray(y).astype(int)
    g = np.asarray(groups_frog, dtype=object)
    file_names = np.array(file_names, dtype=object) if file_names is not None else np.array([""] * len(y), dtype=object)

    outer_logo = LeaveOneGroupOut()

    outer_rows = []
    outer_threshold_rows = []

    for outer_i, (tr_idx, te_idx) in enumerate(outer_logo.split(E, y, g)):
        test_frog = np.unique(g[te_idx])
        if len(test_frog) != 1:
            raise ValueError(f"Expected exactly one outer test frog, got {test_frog}")
        test_frog = str(test_frog[0])

        if verbose_outer:
            print("=" * 90)
            print(f"[OUTER {outer_i+1}] test_frog = {test_frog}")

        E_outer_tr = E[tr_idx]
        y_outer_tr = y[tr_idx]
        g_outer_tr = g[tr_idx]

        E_outer_te = E[te_idx]
        y_outer_te = y[te_idx]
        g_outer_te = g[te_idx]
        fn_outer_te = file_names[te_idx]

        # --------------------------------------------------
        # inner LOGO on outer-train to get inner OOF frog probs
        # --------------------------------------------------
        inner_logo = LeaveOneGroupOut()

        inner_prob_file_all = []
        inner_y_file_all = []
        inner_g_file_all = []

        for inner_i, (itr_idx, ival_idx) in enumerate(inner_logo.split(E_outer_tr, y_outer_tr, g_outer_tr)):
            val_frog = np.unique(g_outer_tr[ival_idx])
            if len(val_frog) != 1:
                raise ValueError(f"Expected exactly one inner val frog, got {val_frog}")
            val_frog = str(val_frog[0])

            if verbose_inner:
                print(f"   [INNER {inner_i+1}] val_frog = {val_frog}")

            if len(np.unique(y_outer_tr[itr_idx])) < 2:
                # 极端情况下，inner-train 只有单类，则跳过这个inner fold
                if verbose_inner:
                    print("   skip inner fold due to single-class training set")
                continue

            clf_inner = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    C=C,
                    max_iter=max_iter,
                    class_weight="balanced",
                    solver="lbfgs",
                ),
            )
            clf_inner.fit(E_outer_tr[itr_idx], y_outer_tr[itr_idx])

            p_inner_val = clf_inner.predict_proba(E_outer_tr[ival_idx])[:, 1]

            inner_prob_file_all.append(p_inner_val)
            inner_y_file_all.append(y_outer_tr[ival_idx])
            inner_g_file_all.append(g_outer_tr[ival_idx])

        if len(inner_prob_file_all) == 0:
            raise ValueError(f"All inner folds skipped for outer test frog {test_frog}")

        inner_prob_file_all = np.concatenate(inner_prob_file_all)
        inner_y_file_all = np.concatenate(inner_y_file_all)
        inner_g_file_all = np.concatenate(inner_g_file_all)

        # file -> frog
        inner_prob_frog, inner_frogs = aggregate_by_group(inner_prob_file_all, inner_g_file_all, mode=frog_agg)
        inner_y_frog, inner_frogs_y = aggregate_frog_labels(inner_y_file_all, inner_g_file_all)

        assert np.array_equal(inner_frogs, inner_frogs_y)

        thr, thr_score = choose_best_threshold(
            y_true=inner_y_frog,
            probs=inner_prob_frog,
            metric=threshold_metric,
        )

        if verbose_outer:
            print(f"  chosen threshold = {thr:.6f} | inner_{threshold_metric} = {thr_score:.4f}")

        outer_threshold_rows.append({
            "outer_test_frog": test_frog,
            "threshold": thr,
            f"inner_{threshold_metric}": thr_score,
            "n_inner_frogs": len(inner_y_frog),
        })

        # --------------------------------------------------
        # final fit on full outer-train
        # --------------------------------------------------
        if len(np.unique(y_outer_tr)) < 2:
            raise ValueError(f"Outer train set has single class for outer test frog {test_frog}")

        clf_final = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=C,
                max_iter=max_iter,
                class_weight="balanced",
                solver="lbfgs",
            ),
        )
        clf_final.fit(E_outer_tr, y_outer_tr)

        p_outer_test_file = clf_final.predict_proba(E_outer_te)[:, 1]

        # file -> frog for outer test
        p_outer_test_frog, test_frogs2 = aggregate_by_group(p_outer_test_file, g_outer_te, mode=frog_agg)
        y_outer_test_frog, test_frogs3 = aggregate_frog_labels(y_outer_te, g_outer_te)

        assert np.array_equal(test_frogs2, test_frogs3)
        assert len(p_outer_test_frog) == 1

        prob_frog = float(p_outer_test_frog[0])
        y_frog = int(y_outer_test_frog[0])
        pred_frog = int(prob_frog >= thr)

        outer_rows.append({
            "frog_id": test_frog,
            "y": y_frog,
            "prob": prob_frog,
            "pred": pred_frog,
            "threshold": thr,
            f"inner_{threshold_metric}": thr_score,
            "n_files": len(te_idx),
            "file_names": "|".join(map(str, fn_outer_te)),
        })

    df_frog = pd.DataFrame(outer_rows).sort_values("frog_id").reset_index(drop=True)
    df_thr = pd.DataFrame(outer_threshold_rows).sort_values("outer_test_frog").reset_index(drop=True)

    Y = df_frog["y"].values.astype(int)
    P = df_frog["prob"].values.astype(float)
    Pred = df_frog["pred"].values.astype(int)

    metrics = {
        "ACC_tuned": float(accuracy_score(Y, Pred)),
        "Balanced_ACC_tuned": float(balanced_accuracy_score(Y, Pred)),
        "AUROC": float(roc_auc_score(Y, P)) if len(np.unique(Y)) >= 2 else np.nan,
        "AUPRC": float(average_precision_score(Y, P)) if len(np.unique(Y)) >= 2 else np.nan,
        "n_frogs": int(len(Y)),
        "n_pos_frogs": int(Y.sum()),
        "n_neg_frogs": int((1 - Y).sum()),
    }

    return {
        "metrics": metrics,
        "df_frog": df_frog,
        "df_outer_thresholds": df_thr,
    }


# ============================================================
# 8) FIND BEST CKPT PER SEED
# ============================================================

def find_best_ckpt_for_seed(best_ckpt_dir: Path, seed: int):
    cands = sorted(best_ckpt_dir.glob(f"seed{seed}_*.ckpt"))
    if len(cands) == 0:
        raise FileNotFoundError(f"No checkpoint found for seed={seed} in {best_ckpt_dir}")
    if len(cands) > 1:
        print(f"[WARN] multiple checkpoints found for seed={seed}; use first: {cands[0].name}")
    return str(cands[0])


# ============================================================
# 9) BUILD OR LOAD FILE EMBEDDINGS FOR ONE SEED
# ============================================================

def build_or_load_file_embeddings_for_seed(
    seed,
    best_ckpt_dir,
    npz_dir,
    meta_path,
    device,
    pos_class,
    neg_class="control",
    save_cache=True,
    embed_cache_dir=None,
    window_size=500,
    stride=250,
    embed_dim=64,
    patch_size=50,
    encoder_stride=50,
    num_heads=4,
):
    embed_cache_dir = Path(embed_cache_dir)
    embed_cache_dir.mkdir(parents=True, exist_ok=True)

    cache_csv = embed_cache_dir / f"file_embeddings__{pos_class}_vs_{neg_class}__seed{seed}.csv"

    if save_cache and cache_csv.exists():
        df_embed = pd.read_csv(cache_csv)
        emb_cols = [c for c in df_embed.columns if c.startswith("emb_")]
        print(f"[CACHE] loaded {cache_csv}")
        print(df_embed.shape)
        return df_embed, emb_cols

    npz_files, eval_loader = build_eval_loader(
        npz_dir=npz_dir,
        seed=seed,
        window_size=window_size,
        stride=stride,
    )

    y_file_str_map, _, _ = build_npz_to_label_maps(npz_files, meta_path)

    ckpt_path = find_best_ckpt_for_seed(best_ckpt_dir, seed)
    encoder = load_encoder_from_ckpt(
        ckpt_path=ckpt_path,
        device=device,
        embed_dim=embed_dim,
        patch_size=patch_size,
        encoder_stride=encoder_stride,
        num_heads=num_heads,
    )

    E_file, y_file_str, g_file, file_names = extract_E_file_from_encoder(
        encoder, eval_loader, device, y_file_str_map
    )

    # keep only task files
    E2, y2, y2_str, g2, fn2 = filter_task_files(
        E_file=E_file,
        y_file_str=y_file_str,
        g_file=g_file,
        file_names=file_names,
        pos_class=pos_class,
        neg_class=neg_class,
    )

    rows = []
    for i in range(len(fn2)):
        row = {
            "seed": seed,
            "ckpt_path": ckpt_path,
            "ckpt_name": Path(ckpt_path).name,
            "step": parse_step_from_ckpt_name(Path(ckpt_path).name),
            "file_name": str(fn2[i]),
            "frog_id": str(g2[i]),
            "label_str": str(y2_str[i]),
            "y": int(y2[i]),
        }
        for j, v in enumerate(E2[i]):
            row[f"emb_{j}"] = float(v)
        rows.append(row)

    df_embed = pd.DataFrame(rows)
    emb_cols = [c for c in df_embed.columns if c.startswith("emb_")]

    if save_cache:
        df_embed.to_csv(cache_csv, index=False)
        print(f"[CACHE] saved {cache_csv}")

    return df_embed, emb_cols


# ============================================================
# 10) RUN ONE SEED
# ============================================================

def run_one_seed_nested_eval(
    seed,
    pos_class,
    neg_class="control",
):
    df_embed, emb_cols = build_or_load_file_embeddings_for_seed(
        seed=seed,
        best_ckpt_dir=BEST_CKPT_DIR,
        npz_dir=NPZ_DIR,
        meta_path=META_PATH,
        device=DEVICE,
        pos_class=pos_class,
        neg_class=neg_class,
        save_cache=SAVE_EMBED_CACHE,
        embed_cache_dir=EMBED_CACHE_DIR,
        window_size=WINDOW_SIZE,
        stride=STRIDE,
        embed_dim=EMBED_DIM,
        patch_size=PATCH_SIZE,
        encoder_stride=ENCODER_STRIDE,
        num_heads=NUM_HEADS,
    )

    E = df_embed[emb_cols].values
    y = df_embed["y"].values.astype(int)
    g = df_embed["frog_id"].values.astype(str)
    fn = df_embed["file_name"].values.astype(object)

    res = run_strict_nested_logo_threshold_tuning(
        E=E,
        y=y,
        groups_frog=g,
        file_names=fn,
        C=LR_C,
        frog_agg=FROG_AGG,
        threshold_metric=THRESHOLD_TUNE_METRIC,
        max_iter=LR_MAX_ITER,
        verbose_outer=VERBOSE_OUTER,
        verbose_inner=VERBOSE_INNER,
    )

    metrics_row = {
        "seed": seed,
        "task": f"{pos_class} vs {neg_class}",
        "ckpt_path": df_embed["ckpt_path"].iloc[0],
        "ckpt_name": df_embed["ckpt_name"].iloc[0],
        "step": df_embed["step"].iloc[0],
        "threshold_metric": THRESHOLD_TUNE_METRIC,
        "frog_agg": FROG_AGG,
        **res["metrics"],
    }

    return metrics_row, res, df_embed


# ============================================================
# 11) TEST A SINGLE SEED FIRST
# ============================================================

if RUN_ONE_SEED is not None:
    metrics_row_1, res_1, df_embed_1 = run_one_seed_nested_eval(
        seed=RUN_ONE_SEED,
        pos_class=POS_CLASS,
        neg_class=NEG_CLASS,
    )

    print("\nSingle-seed metrics:")
    print(metrics_row_1)

    display(df_embed_1.head())
    display(pd.DataFrame([metrics_row_1]))
    display(res_1["df_frog"])
    display(res_1["df_outer_thresholds"])

    out_seed_dir = OUT_DIR / f"{POS_CLASS}_vs_{NEG_CLASS}" / f"seed{RUN_ONE_SEED}"
    out_seed_dir.mkdir(parents=True, exist_ok=True)

    pd.DataFrame([metrics_row_1]).to_csv(out_seed_dir / "metrics.csv", index=False)
    res_1["df_frog"].to_csv(out_seed_dir / "frog_predictions.csv", index=False)
    res_1["df_outer_thresholds"].to_csv(out_seed_dir / "outer_thresholds.csv", index=False)

    print("Saved single-seed outputs to:", out_seed_dir)


# ============================================================
# 12) RUN ALL SEEDS
# ============================================================

def run_all_seeds_nested_eval(
    seeds,
    pos_class,
    neg_class="control",
):
    all_metrics = []
    all_frog_tables = []
    all_thr_tables = []

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"RUNNING SEED {seed}")
        print("#" * 100)

        metrics_row, res, _ = run_one_seed_nested_eval(
            seed=seed,
            pos_class=pos_class,
            neg_class=neg_class,
        )

        all_metrics.append(metrics_row)

        df_frog = res["df_frog"].copy()
        df_frog["seed"] = seed
        all_frog_tables.append(df_frog)

        df_thr = res["df_outer_thresholds"].copy()
        df_thr["seed"] = seed
        all_thr_tables.append(df_thr)

        out_seed_dir = OUT_DIR / f"{pos_class}_vs_{neg_class}" / f"seed{seed}"
        out_seed_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame([metrics_row]).to_csv(out_seed_dir / "metrics.csv", index=False)
        df_frog.to_csv(out_seed_dir / "frog_predictions.csv", index=False)
        df_thr.to_csv(out_seed_dir / "outer_thresholds.csv", index=False)

    results_per_seed = pd.DataFrame(all_metrics).sort_values("seed").reset_index(drop=True)
    all_frog_predictions = pd.concat(all_frog_tables, axis=0, ignore_index=True)
    all_outer_thresholds = pd.concat(all_thr_tables, axis=0, ignore_index=True)

    return results_per_seed, all_frog_predictions, all_outer_thresholds


# ============================================================
# 13) OPTIONAL: RUN ALL
# ============================================================

# 先确认单个 seed 没问题，再取消下面注释运行全部
results_per_seed, all_frog_predictions, all_outer_thresholds = run_all_seeds_nested_eval(
    seeds=SEEDS,
    pos_class=POS_CLASS,
    neg_class=NEG_CLASS,
)

display(results_per_seed)
display(all_frog_predictions.head())
display(all_outer_thresholds.head())


# ============================================================
# 14) SAVE / SUMMARY HELPERS
# ============================================================

def save_all_outputs(results_per_seed, all_frog_predictions, all_outer_thresholds, pos_class, neg_class="control"):
    task_out_dir = OUT_DIR / f"{pos_class}_vs_{neg_class}"
    task_out_dir.mkdir(parents=True, exist_ok=True)

    out_metrics = task_out_dir / "nested_logo_threshold__metrics_per_seed.csv"
    out_frogs   = task_out_dir / "nested_logo_threshold__all_frog_predictions.csv"
    out_thr     = task_out_dir / "nested_logo_threshold__all_outer_thresholds.csv"

    results_per_seed.to_csv(out_metrics, index=False)
    all_frog_predictions.to_csv(out_frogs, index=False)
    all_outer_thresholds.to_csv(out_thr, index=False)

    print("Saved:")
    print(out_metrics)
    print(out_frogs)
    print(out_thr)

def mean_std_str(x):
    x = pd.Series(x).dropna().astype(float)
    return f"{x.mean():.3f} ± {x.std(ddof=1):.3f}"

def mean_ci_str(x):
    x = pd.Series(x).dropna().astype(float)
    n = len(x)
    mean = x.mean()
    std = x.std(ddof=1)
    se = std / np.sqrt(n) if n > 0 else np.nan
    lo = mean - 1.96 * se
    hi = mean + 1.96 * se
    return f"{mean:.3f} ({lo:.3f}–{hi:.3f})"

def make_summary_tables(results_per_seed, pos_class, neg_class="control"):
    summary_mean_std = pd.DataFrame([{
        "task": f"{pos_class} vs {neg_class}",
        "n_seeds": len(results_per_seed),
        "ACC_tuned": mean_std_str(results_per_seed["ACC_tuned"]),
        "Balanced_ACC_tuned": mean_std_str(results_per_seed["Balanced_ACC_tuned"]),
        "AUROC": mean_std_str(results_per_seed["AUROC"]),
        "AUPRC": mean_std_str(results_per_seed["AUPRC"]),
    }])

    summary_mean_ci = pd.DataFrame([{
        "task": f"{pos_class} vs {neg_class}",
        "n_seeds": len(results_per_seed),
        "ACC_tuned": mean_ci_str(results_per_seed["ACC_tuned"]),
        "Balanced_ACC_tuned": mean_ci_str(results_per_seed["Balanced_ACC_tuned"]),
        "AUROC": mean_ci_str(results_per_seed["AUROC"]),
        "AUPRC": mean_ci_str(results_per_seed["AUPRC"]),
    }])

    return summary_mean_std, summary_mean_ci


# ============================================================
# 15) AFTER RUN ALL, USE THESE
# ============================================================

save_all_outputs(results_per_seed, all_frog_predictions, all_outer_thresholds, POS_CLASS, NEG_CLASS)
summary_mean_std, summary_mean_ci = make_summary_tables(results_per_seed, POS_CLASS, NEG_CLASS)
display(summary_mean_std)
display(summary_mean_ci)